# Milestone 4 - Recommendation Prototype

## Objective

Turn model predictions into actionable advice for campaigns predicted to fail. The prototype produces **two recommendations** for the same campaign:

| | Plan A: pre-launch advice | Plan B: with `updates` |
|---|---|---|
| Model | Tuned XGBoost, **no** `updates` | Tuned XGBoost, **with** `updates` |
| Levers it recommends changing | funding goal, campaign length, reward levels | the same three (scored by a model that also knows the campaign's updates) |
| Extra output | none | An engagement note comparing the campaign's updates with successful peers |

**How it works.** For a campaign the model expects to fail, the code changes one lever at a time to realistic values, re-scores the campaign with the model, and ranks the changes by how much they raise the predicted chance of success. It then shows the best single change for each lever and the predicted result of applying them together.

**Why updates are context, not a lever.** In a first version, Plan B also recommended posting more updates. That suggestion dominated every result (it was the top suggestion for about 99% of campaigns, with predicted lifts of +60 points). That is not credible as cause and effect: campaigns that are going well tend to post more updates *because* they are going well. So Plan B uses `updates` only as **context** (it makes the predicted chance of success much more informative) and recommends changes only to goal, length, and reward levels. Updates appear as a clearly labeled **engagement note**, never as a promised fix.

**Important limits.** These are *model estimates of association*, not guarantees. The model learned patterns from past campaigns; it cannot prove that making a change will cause a campaign to succeed.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

pd.set_option("display.width", 160)

## 1. Data and the two models

Same data, target, and 80/20 stratified split (`random_state=42`) as Tasks 1-8. Both models use the tuned XGBoost settings from Task 5 (`learning_rate=0.1`, `max_depth=3`, `n_estimators=400`). They differ only in whether `updates` is a feature.

In [2]:
df = pd.read_csv("../data/kickstarter_us_filtered.csv")
df["success"] = (df["status"] == "successful").astype(int)
df["state"] = df["location"].str.rsplit(",", n=1).str[-1].str.strip()

CAT = ["category", "subcategory", "state"]
NUM_A = ["goal", "duration", "levels"]               # Plan A: pre-launch
NUM_B = ["goal", "duration", "levels", "updates"]    # Plan B: with updates

y = df["success"]
train_idx, test_idx = train_test_split(df.index, test_size=0.20, random_state=42, stratify=y)
train, test = df.loc[train_idx], df.loc[test_idx]
print("Train:", train.shape, "| Test:", test.shape)

def make_model(num_features):
    pre = ColumnTransformer([
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
    ])
    return Pipeline([
        ("preprocessor", pre),
        ("model", XGBClassifier(random_state=42, eval_metric="logloss",
                                learning_rate=0.1, max_depth=3, n_estimators=400)),
    ])

model_a = make_model(NUM_A).fit(train[NUM_A + CAT], train["success"])
model_b = make_model(NUM_B).fit(train[NUM_B + CAT], train["success"])

for name, m, num in [("Plan A (no updates)", model_a, NUM_A), ("Plan B (with updates)", model_b, NUM_B)]:
    p = m.predict_proba(test[num + CAT])[:, 1]
    print(f"{name}: accuracy={accuracy_score(test['success'], p > 0.5):.3f}  ROC-AUC={roc_auc_score(test['success'], p):.3f}")

Train: (30803, 19) | Test: (7701, 19)


Plan A (no updates): accuracy=0.703  ROC-AUC=0.767
Plan B (with updates): accuracy=0.814  ROC-AUC=0.898


## 2. Are the predicted probabilities trustworthy?

Recommendations are based on changes in predicted probability, so the probabilities should be roughly calibrated: of campaigns given a predicted chance near 30%, about 30% should actually succeed.

In [3]:
def calibration_table(model, num):
    p = model.predict_proba(test[num + CAT])[:, 1]
    bins = pd.cut(p, [0, .2, .4, .6, .8, 1.0], include_lowest=True)
    out = (pd.DataFrame({"bin": bins, "pred": p, "actual": test["success"].values})
           .groupby("bin", observed=True).agg(campaigns=("pred", "size"),
                                              mean_predicted=("pred", "mean"),
                                              actual_success=("actual", "mean")))
    return out.round(3)

print("Plan A (no updates)"); print(calibration_table(model_a, NUM_A)); print()
print("Plan B (with updates)"); print(calibration_table(model_b, NUM_B))

Plan A (no updates)
               campaigns  mean_predicted  actual_success
bin                                                     
(-0.001, 0.2]        819           0.114           0.106
(0.2, 0.4]          1261           0.307           0.312
(0.4, 0.6]          1984           0.509           0.510
(0.6, 0.8]          2628           0.703           0.711
(0.8, 1.0]          1009           0.847           0.847

Plan B (with updates)
               campaigns  mean_predicted  actual_success
bin                                                     
(-0.001, 0.2]       1993           0.080           0.073
(0.2, 0.4]           977           0.295           0.311
(0.4, 0.6]           853           0.502           0.545
(0.6, 0.8]          1081           0.708           0.691
(0.8, 1.0]          2797           0.914           0.913


## 3. The recommendation engine

**Levers and the values tried** (kept within realistic ranges seen in the training data):

- **Goal:** 90%, 80%, 67%, and 50% of the current goal (never below the 1st percentile of goals in the data).
- **Campaign length:** 20, 30, 45, or 60 days.
- **Reward levels:** current + 2, + 4, or + 6 (capped at the 90th percentile, to keep suggestions realistic).
- **Updates:** not recommended as a lever (see the note at the top). An optional switch, `suggest_updates=True`, exists only to demonstrate the problem.

Category, subcategory, and state are held fixed, since changing a project's category or location is not a realistic suggestion.

In [4]:
GOAL_FLOOR = train["goal"].quantile(0.01)
LEVELS_CAP = int(train["levels"].quantile(0.90))
UPDATES_CAP = int(train["updates"].quantile(0.99))

def candidate_changes(row, suggest_updates=False):
    """Return a list of (lever, description, {column: new_value}) for one campaign."""
    out = []
    g = float(row["goal"])
    for f in [0.9, 0.8, 0.67, 0.5]:
        new = max(round(g * f, -1) if g >= 100 else round(g * f), GOAL_FLOOR)
        if new < g:
            out.append(("goal", f"Lower the funding goal from {g:,.0f} to {new:,.0f} USD", {"goal": float(new)}))
    for d in [20, 30, 45, 60]:
        if abs(d - float(row["duration"])) >= 1:
            out.append(("duration", f"Set the campaign length to {d} days (currently {row['duration']:.0f})", {"duration": float(d)}))
    for k in [2, 4, 6]:
        new = min(int(row["levels"]) + k, LEVELS_CAP)
        if new > row["levels"]:
            out.append(("levels", f"Offer {new} reward levels (currently {int(row['levels'])})", {"levels": new}))
    if suggest_updates:
        for k in [3, 6, 10]:
            new = min(int(row["updates"]) + k, UPDATES_CAP)
            if new > row["updates"]:
                out.append(("updates", f"Plan about {new} updates (currently {int(row['updates'])})", {"updates": new}))
    return out

def recommend(campaign, model, num_features, suggest_updates=False, top_n=3, min_lift=0.005):
    """campaign: a one-row Series with the model's input columns."""
    cols = num_features + CAT
    base = campaign[cols].to_frame().T.astype({c: float for c in num_features})
    base_p = float(model.predict_proba(base)[:, 1][0])

    cands = candidate_changes(campaign, suggest_updates)
    if not cands:
        return {"current_probability": base_p, "recommendations": [], "combined_probability": base_p}
    rows = []
    for lever, text, change in cands:
        r = base.copy()
        for k, v in change.items():
            r[k] = float(v)
        rows.append(r)
    probs = model.predict_proba(pd.concat(rows, ignore_index=True))[:, 1]

    table = pd.DataFrame({"lever": [c[0] for c in cands], "change": [c[1] for c in cands],
                          "new_values": [c[2] for c in cands], "new_probability": probs})
    table["lift"] = table["new_probability"] - base_p
    best_per_lever = (table.sort_values("lift", ascending=False)
                           .drop_duplicates("lever")
                           .query("lift >= @min_lift")
                           .head(top_n))

    # Apply the best change from each lever together
    combo = base.copy()
    for nv in best_per_lever["new_values"]:
        for k, v in nv.items():
            combo[k] = float(v)
    combined_p = float(model.predict_proba(combo)[:, 1][0]) if len(best_per_lever) else base_p

    return {"current_probability": base_p, "recommendations": best_per_lever, "combined_probability": combined_p}

## 4. Peer benchmark (sanity check)

A recommendation is more convincing if it points toward what successful campaigns in the same area actually look like. This helper reports the typical values for successful campaigns in the same subcategory (or category if the subcategory is small).

In [5]:
def peer_benchmark(campaign, min_n=30):
    peers = train[(train["subcategory"] == campaign["subcategory"]) & (train["success"] == 1)]
    label = campaign["subcategory"]
    if len(peers) < min_n:
        peers = train[(train["category"] == campaign["category"]) & (train["success"] == 1)]
        label = campaign["category"]
    return label, len(peers), peers[["goal", "duration", "levels", "updates"]].median()

## 5. Demo: one campaign, two recommendations

In [6]:
def show_plans(campaign):
    print("=" * 78)
    print(f"Campaign: {campaign['category']} / {campaign['subcategory']} ({campaign['state']})")
    print(f"  Goal {campaign['goal']:,.0f} USD | {campaign['duration']:.0f} days | {int(campaign['levels'])} reward levels | {int(campaign['updates'])} updates")
    print(f"  Actual outcome (from the data): {campaign['status']}")
    for title, model, num, use_u in [("PLAN A - pre-launch advice (no updates)", model_a, NUM_A, False),
                                     ("PLAN B - with updates as context", model_b, NUM_B, False)]:
        res = recommend(campaign, model, num, use_u)
        print(f"\n{title}")
        print(f"  Predicted chance of success today: {res['current_probability']:.0%}")
        if len(res["recommendations"]) == 0:
            print("  No change raised the predicted chance meaningfully.")
            continue
        for i, (_, r) in enumerate(res["recommendations"].iterrows(), 1):
            print(f"  {i}. {r['change']}  ->  {r['new_probability']:.0%} ({r['lift']:+.0%})")
        print(f"  All of the above together (optimistic upper bound): about {res['combined_probability']:.0%}")
    label, n, med = peer_benchmark(campaign)
    print(f"\nTypical SUCCESSFUL campaign in '{label}' (n={n}): goal {med['goal']:,.0f} USD, {med['duration']:.0f} days, {med['levels']:.0f} levels, {med['updates']:.0f} updates")
    print(f"ENGAGEMENT NOTE (association, not a promise): successful peers typically post about {med['updates']:.0f} updates; this campaign has {int(campaign['updates'])}.")

p_a = model_a.predict_proba(test[NUM_A + CAT])[:, 1]
p_b = model_b.predict_proba(test[NUM_B + CAT])[:, 1]
at_risk = test[(p_a < 0.5) & (p_b < 0.5) & (p_a > 0.25)]   # both models expect failure
demo = at_risk.sample(3, random_state=7)
for _, camp in demo.iterrows():
    show_plans(camp)

Campaign: Film &amp; Video / Documentary (GA)
  Goal 3,500 USD | 45 days | 8 reward levels | 0 updates
  Actual outcome (from the data): failed

PLAN A - pre-launch advice (no updates)
  Predicted chance of success today: 48%
  1. Offer 13 reward levels (currently 8)  ->  59% (+11%)
  2. Set the campaign length to 20 days (currently 45)  ->  58% (+10%)
  3. Lower the funding goal from 3,500 to 1,750 USD  ->  57% (+9%)
  All of the above together (optimistic upper bound): about 71%



PLAN B - with updates as context
  Predicted chance of success today: 10%
  1. Lower the funding goal from 3,500 to 1,750 USD  ->  17% (+7%)
  2. Set the campaign length to 20 days (currently 45)  ->  17% (+7%)
  All of the above together (optimistic upper bound): about 31%

Typical SUCCESSFUL campaign in 'Documentary' (n=1204): goal 6,000 USD, 35 days, 8 levels, 6 updates
ENGAGEMENT NOTE (association, not a promise): successful peers typically post about 6 updates; this campaign has 0.
Campaign: Photography / Photography (NY)
  Goal 5,000 USD | 45 days | 5 reward levels | 0 updates
  Actual outcome (from the data): successful

PLAN A - pre-launch advice (no updates)
  Predicted chance of success today: 44%
  1. Offer 9 reward levels (currently 5)  ->  62% (+18%)
  2. Lower the funding goal from 5,000 to 2,500 USD  ->  60% (+16%)
  3. Set the campaign length to 20 days (currently 45)  ->  54% (+9%)
  All of the above together (optimistic upper bound): about 77%

PLAN B - with updates 

## 6. How much do the recommendations help, across many at-risk campaigns?

For a sample of test-set campaigns that each model predicts will fail, find the single best recommended change and measure the predicted lift. This shows which lever matters most and how large the typical effect is.

In [7]:
def summarize(model, num, use_u, n=300):
    p = model.predict_proba(test[num + CAT])[:, 1]
    pool = test[p < 0.5]
    sample = pool.sample(min(n, len(pool)), random_state=1)
    rows = []
    for _, camp in sample.iterrows():
        res = recommend(camp, model, num, use_u)
        recs = res["recommendations"]
        if len(recs):
            top = recs.iloc[0]
            rows.append({"top_lever": top["lever"], "top_lift": top["lift"],
                         "combined_lift": res["combined_probability"] - res["current_probability"]})
        else:
            rows.append({"top_lever": "none", "top_lift": 0.0, "combined_lift": 0.0})
    r = pd.DataFrame(rows)
    return r

for name, m, num, use_u in [("Plan A (no updates)", model_a, NUM_A, False), ("Plan B (with updates as context)", model_b, NUM_B, False)]:
    r = summarize(m, num, use_u)
    print(name, f"- sampled {len(r)} campaigns predicted to fail")
    print(f"  Median lift from the best single change: {r['top_lift'].median():+.1%}")
    print(f"  Median lift from combining the best change per lever (optimistic upper bound): {r['combined_lift'].median():+.1%}")
    print("  Which lever was the top suggestion:")
    print((r['top_lever'].value_counts(normalize=True).round(2)).to_string())
    print()

Plan A (no updates) - sampled 300 campaigns predicted to fail
  Median lift from the best single change: +18.8%
  Median lift from combining the best change per lever (optimistic upper bound): +32.3%
  Which lever was the top suggestion:
top_lever
levels      0.67
goal        0.18
duration    0.15



Plan B (with updates as context) - sampled 300 campaigns predicted to fail
  Median lift from the best single change: +10.7%
  Median lift from combining the best change per lever (optimistic upper bound): +21.6%
  Which lever was the top suggestion:
top_lever
goal        0.50
duration    0.39
levels      0.11
none        0.01



## 7. Limitations and next steps

- **Association, not causation.** Recommendations come from patterns in historical data. Lowering a goal or adding reward levels is associated with higher predicted success, but this prototype cannot prove it would change an individual campaign's outcome.
- **Updates.** Updates may partly be a *result* of a campaign doing well. When we allowed the model to recommend more updates, it dominated everything with implausibly large lifts, so updates are used only as context and a labeled engagement note.
- **Combined estimates are optimistic.** The "all together" number assumes each change's effect adds up and that the model extrapolates well; treat it as an upper bound.
- **One lever at a time.** Single-lever changes are scored independently; the combined estimate assumes the effects add up, which may not hold exactly.
- **Fixed attributes.** Category, subcategory, and state are not changed. A natural extension is to suggest a better-fitting subcategory.
- **Possible improvements:** calibrate the probabilities if Section 2 shows large gaps, add a lower decision threshold so more at-risk campaigns are flagged, and wrap `recommend()` in a small web demo (for example Streamlit).